In [ ]:
import pandas as pd

import great_expectations as gx
import synapseclient

from agoradatatools.gx import GreatExpectationsRunner

context = gx.get_context(project_root_dir='../src/agoradatatools/great_expectations')

from expectations.expect_column_nested_field_values_mostly_meet_string_requirement import ExpectColumnMostlyStringLength
    

# Create Expectation Suite for UI Config Data

## Get Example Data File

In [ ]:
syn = synapseclient.Synapse()
syn.login()

In [ ]:
ui_config_data_file = syn.get("syn66531901").path

## Create Validator Object on Data File

In [ ]:
df = pd.read_json(ui_config_data_file)
nested_columns = ["columns"]
df = GreatExpectationsRunner.convert_nested_columns_to_json(df, nested_columns)
validator = context.sources.pandas_default.read_dataframe(df)
validator.expectation_suite_name = "ui_config"

## Add Expectations to Validator Object For Each Column

In [ ]:
validator.expect_column_mostly_string_length(column="columns", target_field="tooltip", operator=">", length_threshold=0, valid_threshold=0.35)

## Save Expectation Suite

In [ ]:
validator.save_expectation_suite(discard_failed_expectations=False)

## Create Checkpoint and View Results

In [ ]:
checkpoint = context.add_or_update_checkpoint(
    name="agora-test-checkpoint",
    validator=validator,
)
checkpoint_result = checkpoint.run()
context.view_validation_result(checkpoint_result)

## Build Data Docs - Click on Expectation Suite to View All Expectations

In [ ]:
context.build_data_docs()
context.open_data_docs()